## 注意

- `Chroma.from_documents` 对同一个 `collection_name` **重复调用会追加，不会覆盖**
- 每次跑脚本都调用会导致重复写入，检索结果出现大量重复内容
- 建库完成后应改用 `Chroma(persist_directory=..., embedding_function=...)` 只加载，不再 `from_documents`
- 需要重建时：先删除 `data/chroma_db` 目录，再重新跑一次

---

```
from_documents  →  embedding=           （建库，只跑一次）
Chroma(...)     →  embedding_function=  （加载已有库）
```

In [ ]:
import os
print(os.environ.get("HF_ENDPOINT"))  # 如果是 None，说明内核没读到

In [ ]:
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from pypdf import PdfReader


def load_pdf(file_path: str | Path) -> list[Document]:
    """将 PDF 按页加载为 LangChain Document 列表。

    Args:
        file_path: PDF 文件路径。

    Returns:
        每页对应一个 Document，metadata 含 source 与 page。
    """
    path = Path(file_path)
    reader = PdfReader(str(path))
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": str(path), "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

In [ ]:
CHROMA_DIR = Path("data/chroma_db")
PDF_PATH = Path("data/员工手册.pdf")

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# ~/.cache/huggingface/hub/
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5",
    # model_kwargs={"local_files_only": True},  # 使用本地模型，前提是下载了模型
)

### 建库

In [ ]:
docs = load_pdf(PDF_PATH)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)
chunks = splitter.split_documents(docs)

vectorstore = Chroma.from_documents(
    collection_name="employee_handbook",  # 集合名，同一目录可存多个集合
    documents=chunks,                     # 分块后的文本列表
    embedding=embeddings,                 # 向量化模型，把每块文字转成向量
    persist_directory=str(CHROMA_DIR),    # 向量库持久化目录，重启后可直接加载
)

### 使用

In [ ]:
vectorstore = Chroma(
    collection_name="employee_handbook",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

hits = retriever.invoke("什么时候发工资")

print(len(hits))

with open(Path("data/rag_result_demo.txt"), "w", encoding="utf-8") as f:
    f.writelines([i.page_content + "\n" + "-" * 20 + "\n" for i in hits])